In [5]:
# --- Jupyter cell: ops.npy modifier based on Excel mapping ---
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

def update_ops_from_excel(
    excel_path: str,
    binary_subdir: str = "binary",
    binary_filename: str = "CSC_Raw.dat",
    make_backup: bool = True
) -> pd.DataFrame:
    """
    Excel format (each column = one session):
        row 0: session_name
        row 1: session_path (folder that contains ops.npy)
    Behavior:
        - Loads <session_path>/ops.npy
        - Sets:
            ops['data_dir'] = parent(session_path)
            ops['filename'] = parent(session_path) / binary_subdir / binary_filename
        - Saves ops.npy (with optional .bak backup)
    Returns a summary DataFrame.
    """
    df = pd.read_excel(excel_path, header=None)
    summaries = []

    for col in df.columns:
        session_name = str(df.iloc[0, col]).strip()
        session_path_str = str(df.iloc[1, col]).strip()

        if not session_path_str or session_path_str.lower() == 'nan':
            print(f"[SKIP] Column {col}: empty session path")
            continue

        session_path = Path(session_path_str)
        ops_path = session_path / "ops.npy"

        # new targets
        parent_dir = session_path.parent
        new_data_dir = parent_dir
        new_filename = parent_dir / binary_filename

        # sanity checks (warnings only)
        exists_ops = ops_path.exists()
        exists_dat = new_filename.exists()

        try:
            if not exists_ops:
                print(f"[WARN] {session_name}: ops.npy not found at {ops_path}")
                summaries.append({
                    "session": session_name,
                    "ops_path": str(ops_path),
                    "updated": False,
                    "reason": "ops.npy not found",
                    "data_dir_set": str(new_data_dir),
                    "filename_set": str(new_filename),
                    "filename_exists": exists_dat
                })
                continue

            # load, modify
            ops = np.load(ops_path, allow_pickle=True).item()
            # optional backup
            if make_backup:
                backup_path = ops_path.with_name(ops_path.name + ".bak")
                np.save(backup_path, ops)

            ops["data_dir"] = str(new_data_dir)
            ops["filename"] = str(new_filename)

            np.save(ops_path, ops)

            print(f"[OK] {session_name}:")
            print(f"     ops:       {ops_path}")
            print(f"     data_dir = {ops['data_dir']}")
            print(f"     filename = {ops['filename']}")
            if not exists_dat:
                print(f"     [WARN] Binary file not found on disk: {new_filename}")

            summaries.append({
                "session": session_name,
                "ops_path": str(ops_path),
                "updated": True,
                "reason": "",
                "data_dir_set": str(new_data_dir),
                "filename_set": str(new_filename),
                "filename_exists": exists_dat
            })

        except Exception as e:
            print(f"[ERROR] {session_name}: {e}")
            summaries.append({
                "session": session_name,
                "ops_path": str(ops_path),
                "updated": False,
                "reason": str(e),
                "data_dir_set": str(new_data_dir),
                "filename_set": str(new_filename),
                "filename_exists": exists_dat
            })

    summary_df = pd.DataFrame(summaries)
    return summary_df

# --- Example usage ---
# summary = update_ops_from_excel(r"E:\path\to\your\sessions.xlsx")
# display(summary)


In [6]:
summary = update_ops_from_excel(r"D:\Data\PTEN Project\feedersheets\feedersheet_notinvert.xlsx", make_backup='False')

[OK] m1s2:
     ops:       D:\Data\PTEN Project\PTEN\M1_s2\2023-10-02_16-58-03\binary\kilosort_tl6_tu7_fs30000.0\ops.npy
     data_dir = D:\Data\PTEN Project\PTEN\M1_s2\2023-10-02_16-58-03\binary
     filename = D:\Data\PTEN Project\PTEN\M1_s2\2023-10-02_16-58-03\binary\CSC_Raw.dat
[OK] m1s8:
     ops:       D:\Data\PTEN Project\PTEN\M1_s8\2023-10-05_14-09-57\binary\kilosort_tl6_tu7_fs30000.0\ops.npy
     data_dir = D:\Data\PTEN Project\PTEN\M1_s8\2023-10-05_14-09-57\binary
     filename = D:\Data\PTEN Project\PTEN\M1_s8\2023-10-05_14-09-57\binary\CSC_Raw.dat
[OK] m2s3:
     ops:       D:\Data\PTEN Project\Control\M2_s3\2024-01-23_16-04-25\binary\kilosort_tl6_tu7_fs30000.0\ops.npy
     data_dir = D:\Data\PTEN Project\Control\M2_s3\2024-01-23_16-04-25\binary
     filename = D:\Data\PTEN Project\Control\M2_s3\2024-01-23_16-04-25\binary\CSC_Raw.dat
[OK] m2s8:
     ops:       D:\Data\PTEN Project\Control\M2_s8\2024-01-26_17-48-00\binary\kilosort_tl6_tu7_fs30000.0\ops.npy
     data_dir = D: